# Cryptography Practicals — Cipher Implementations

| Prac | Cipher | Type |
|------|--------|------|
| 5 | Caesar Cipher | Substitution |
| 6 | Monoalphabetic & Polyalphabetic (Vigenère) | Substitution |
| 7 | Playfair Cipher | Substitution |
| 8 | Hill Cipher | Substitution |
| 9 | Rail Fence Cipher | Transposition |
| 10 | Row Transposition Cipher | Transposition |
| 11 | Product Cipher | Transposition |

> **Note:** Prac 8 (Hill Cipher) requires `numpy`. Install with: `pip install numpy`

---
## Prac 5 — Caesar Cipher (Substitution)
Each letter is shifted by a fixed number (key) in the alphabet.
- **Encryption:** `C = (P + key) mod 26`
- **Decryption:** `P = (C - key) mod 26`

In [ ]:
# ============================================================
# PRAC 5: Caesar Cipher Substitution Operation
# ============================================================

def caesar_encrypt(plaintext, key):
    result = ''
    for ch in plaintext:
        if ch.isalpha():
            base = ord('A') if ch.isupper() else ord('a')
            result += chr((ord(ch) - base + key) % 26 + base)
        else:
            result += ch   # spaces, digits, punctuation unchanged
    return result

def caesar_decrypt(ciphertext, key):
    return caesar_encrypt(ciphertext, -key)  # shift back by key

def brute_force(ciphertext):
    """Try all 25 possible shifts."""
    print("\nBrute Force (all 25 shifts):")
    print("-" * 40)
    for shift in range(1, 26):
        print(f"  Key {shift:2d}: {caesar_encrypt(ciphertext, -shift)}")

# ==================== DEMO ====================
print("=" * 50)
print("         CAESAR CIPHER")
print("=" * 50)

plaintext = "Hello World"
key = 3   # Classic Caesar uses key = 3

encrypted = caesar_encrypt(plaintext, key)
decrypted = caesar_decrypt(encrypted, key)

print(f"Plaintext  : {plaintext}")
print(f"Key (shift): {key}")
print(f"Encrypted  : {encrypted}")
print(f"Decrypted  : {decrypted}")

print("\n--- Another Example (ROT13, key=13) ---")
plaintext2 = "ATTACKATDAWN"
key2 = 13
encrypted2 = caesar_encrypt(plaintext2, key2)
decrypted2 = caesar_decrypt(encrypted2, key2)
print(f"Plaintext  : {plaintext2}")
print(f"Key (shift): {key2}")
print(f"Encrypted  : {encrypted2}")
print(f"Decrypted  : {decrypted2}")

brute_force(encrypted2)

---
## Prac 6 — Monoalphabetic & Polyalphabetic (Vigenère) Cipher (Substitution)
- **Monoalphabetic:** Fixed shuffled alphabet as the key — every letter always maps to the same cipher letter.
- **Vigenère (Polyalphabetic):** A keyword is repeated; each character shifts by the corresponding keyword letter, so the same plaintext letter encrypts differently at different positions.

In [ ]:
# ============================================================
# PRAC 6: Monoalphabetic and Polyalphabetic (Vigenere) Cipher
# ============================================================

# ---------- MONOALPHABETIC CIPHER ----------
def mono_encrypt(plaintext, key):
    """
    key: a 26-letter string mapping a->key[0], b->key[1], ...
    """
    alphabet = 'abcdefghijklmnopqrstuvwxyz'
    result = ''
    for ch in plaintext.lower():
        if ch in alphabet:
            result += key[alphabet.index(ch)]
        else:
            result += ch
    return result

def mono_decrypt(ciphertext, key):
    alphabet = 'abcdefghijklmnopqrstuvwxyz'
    result = ''
    for ch in ciphertext.lower():
        if ch in key:
            result += alphabet[key.index(ch)]
        else:
            result += ch
    return result


# ---------- POLYALPHABETIC (VIGENERE) CIPHER ----------
def vigenere_encrypt(plaintext, keyword):
    alphabet = 'abcdefghijklmnopqrstuvwxyz'
    keyword = keyword.lower()
    result = ''
    k_idx = 0
    for ch in plaintext.lower():
        if ch in alphabet:
            shift = alphabet.index(keyword[k_idx % len(keyword)])
            result += alphabet[(alphabet.index(ch) + shift) % 26]
            k_idx += 1
        else:
            result += ch
    return result

def vigenere_decrypt(ciphertext, keyword):
    alphabet = 'abcdefghijklmnopqrstuvwxyz'
    keyword = keyword.lower()
    result = ''
    k_idx = 0
    for ch in ciphertext.lower():
        if ch in alphabet:
            shift = alphabet.index(keyword[k_idx % len(keyword)])
            result += alphabet[(alphabet.index(ch) - shift) % 26]
            k_idx += 1
        else:
            result += ch
    return result


# ==================== DEMO ====================
print("=" * 50)
print("   MONOALPHABETIC CIPHER")
print("=" * 50)
mono_key = 'qwertyuiopasdfghjklzxcvbnm'
plaintext = "hello world"
encrypted = mono_encrypt(plaintext, mono_key)
decrypted = mono_decrypt(encrypted, mono_key)
print(f"Plaintext : {plaintext}")
print(f"Key       : {mono_key}")
print(f"Encrypted : {encrypted}")
print(f"Decrypted : {decrypted}")

print("\n" + "=" * 50)
print("   POLYALPHABETIC (VIGENERE) CIPHER")
print("=" * 50)
plaintext = "hello world"
keyword   = "key"
encrypted = vigenere_encrypt(plaintext, keyword)
decrypted = vigenere_decrypt(encrypted, keyword)
print(f"Plaintext : {plaintext}")
print(f"Keyword   : {keyword}")
print(f"Encrypted : {encrypted}")
print(f"Decrypted : {decrypted}")

---
## Prac 7 — Playfair Cipher (Substitution)
A digraph (2-letter block) substitution cipher using a 5×5 key matrix (I and J share a cell).
- **Same row:** shift right
- **Same column:** shift down
- **Rectangle:** swap columns

In [ ]:
# ============================================================
# PRAC 7: Playfair Cipher Substitution
# ============================================================

def generate_matrix(key):
    key = key.upper().replace('J', 'I')
    seen = []
    for ch in key:
        if ch not in seen and ch.isalpha():
            seen.append(ch)
    for ch in 'ABCDEFGHIKLMNOPQRSTUVWXYZ':
        if ch not in seen:
            seen.append(ch)
    matrix = [seen[i*5:(i+1)*5] for i in range(5)]
    return matrix

def find_position(matrix, ch):
    for r in range(5):
        for c in range(5):
            if matrix[r][c] == ch:
                return r, c
    return None

def prepare_text(plaintext):
    text = plaintext.upper().replace('J', 'I')
    text = ''.join(filter(str.isalpha, text))
    result = ''
    i = 0
    while i < len(text):
        a = text[i]
        b = text[i + 1] if i + 1 < len(text) else 'X'
        if a == b:
            result += a + 'X'
            i += 1
        else:
            result += a + b
            i += 2
    if len(result) % 2 != 0:
        result += 'X'
    return result

def playfair_encrypt(plaintext, key):
    matrix = generate_matrix(key)
    text = prepare_text(plaintext)
    ciphertext = ''
    for i in range(0, len(text), 2):
        r1, c1 = find_position(matrix, text[i])
        r2, c2 = find_position(matrix, text[i+1])
        if r1 == r2:                            # same row
            ciphertext += matrix[r1][(c1+1)%5] + matrix[r2][(c2+1)%5]
        elif c1 == c2:                          # same column
            ciphertext += matrix[(r1+1)%5][c1] + matrix[(r2+1)%5][c2]
        else:                                   # rectangle
            ciphertext += matrix[r1][c2] + matrix[r2][c1]
    return ciphertext

def playfair_decrypt(ciphertext, key):
    matrix = generate_matrix(key)
    ciphertext = ciphertext.upper()
    plaintext = ''
    for i in range(0, len(ciphertext), 2):
        r1, c1 = find_position(matrix, ciphertext[i])
        r2, c2 = find_position(matrix, ciphertext[i+1])
        if r1 == r2:
            plaintext += matrix[r1][(c1-1)%5] + matrix[r2][(c2-1)%5]
        elif c1 == c2:
            plaintext += matrix[(r1-1)%5][c1] + matrix[(r2-1)%5][c2]
        else:
            plaintext += matrix[r1][c2] + matrix[r2][c1]
    return plaintext

def print_matrix(matrix):
    print("Playfair Matrix:")
    for row in matrix:
        print(' '.join(row))

# ==================== DEMO ====================
print("=" * 50)
print("   PLAYFAIR CIPHER")
print("=" * 50)
key = "MONARCHY"
plaintext = "INSTRUMENTS"

matrix = generate_matrix(key)
print_matrix(matrix)
print()

encrypted = playfair_encrypt(plaintext, key)
decrypted = playfair_decrypt(encrypted, key)

print(f"Plaintext  : {plaintext}")
print(f"Key        : {key}")
print(f"Prepared   : {prepare_text(plaintext)}")
print(f"Encrypted  : {encrypted}")
print(f"Decrypted  : {decrypted}")

---
## Prac 8 — Hill Cipher (Substitution)
Uses **matrix multiplication mod 26**. The key is an n×n invertible matrix.
- **Encryption:** `C = K × P (mod 26)`
- **Decryption:** `P = K⁻¹ × C (mod 26)`

> Requires `numpy` → `pip install numpy`

In [ ]:
# ============================================================
# PRAC 8: Hill Cipher Substitution
# ============================================================
import numpy as np

def mod_inverse(a, m=26):
    """Find modular inverse of a (mod m)."""
    for x in range(1, m):
        if (a * x) % m == 1:
            return x
    return None

def matrix_mod_inverse(matrix, mod=26):
    """Find modular inverse of a matrix (mod 26)."""
    det = int(round(np.linalg.det(matrix))) % mod
    det_inv = mod_inverse(det % mod, mod)
    if det_inv is None:
        raise ValueError("Matrix is not invertible mod 26")
    size = matrix.shape[0]
    cofactors = np.zeros_like(matrix)
    for r in range(size):
        for c in range(size):
            minor = np.delete(np.delete(matrix, r, axis=0), c, axis=1)
            cofactors[r][c] = ((-1)**(r+c)) * round(np.linalg.det(minor))
    adjugate = cofactors.T
    inv_matrix = (det_inv * adjugate) % mod
    return inv_matrix.astype(int)

def text_to_vector(text):
    return [ord(ch) - ord('A') for ch in text.upper() if ch.isalpha()]

def vector_to_text(vector):
    return ''.join(chr(int(v) % 26 + ord('A')) for v in vector)

def hill_encrypt(plaintext, key_matrix):
    n = key_matrix.shape[0]
    text = ''.join(filter(str.isalpha, plaintext.upper()))
    while len(text) % n != 0:
        text += 'X'
    ciphertext = ''
    for i in range(0, len(text), n):
        block = np.array(text_to_vector(text[i:i+n]))
        encrypted_block = np.dot(key_matrix, block) % 26
        ciphertext += vector_to_text(encrypted_block)
    return ciphertext

def hill_decrypt(ciphertext, key_matrix):
    n = key_matrix.shape[0]
    inv_key = matrix_mod_inverse(key_matrix)
    plaintext = ''
    for i in range(0, len(ciphertext), n):
        block = np.array(text_to_vector(ciphertext[i:i+n]))
        decrypted_block = np.dot(inv_key, block) % 26
        plaintext += vector_to_text(decrypted_block)
    return plaintext

# ==================== DEMO ====================
print("=" * 50)
print("   HILL CIPHER")
print("=" * 50)

# 2x2 key matrix
key_matrix = np.array([[3, 3],
                        [2, 5]])
plaintext = "HELP"
print(f"Key Matrix :\n{key_matrix}")
print(f"Plaintext  : {plaintext}")
encrypted = hill_encrypt(plaintext, key_matrix)
decrypted = hill_decrypt(encrypted, key_matrix)
print(f"Encrypted  : {encrypted}")
print(f"Decrypted  : {decrypted}")

print("\n--- 3x3 Key Matrix Example ---")
key_matrix_3 = np.array([[6,  24, 1],
                          [13, 16, 10],
                          [20, 17, 15]])
plaintext2 = "ACT"
encrypted2 = hill_encrypt(plaintext2, key_matrix_3)
decrypted2 = hill_decrypt(encrypted2, key_matrix_3)
print(f"Key Matrix :\n{key_matrix_3}")
print(f"Plaintext  : {plaintext2}")
print(f"Encrypted  : {encrypted2}")
print(f"Decrypted  : {decrypted2}")

---
## Prac 9 — Rail Fence Cipher (Transposition)
Text is written in a **zigzag** pattern across N rails, then read row by row.
No letters are substituted — only their positions are rearranged.

In [ ]:
# ============================================================
# PRAC 9: Rail Fence Cipher (Transposition)
# ============================================================

def rail_fence_encrypt(plaintext, rails):
    """Write text in zigzag across 'rails' rows, then read row by row."""
    fence = [[] for _ in range(rails)]
    rail = 0
    direction = 1  # 1 = down, -1 = up
    for ch in plaintext:
        fence[rail].append(ch)
        if rail == 0:           direction = 1
        elif rail == rails - 1: direction = -1
        rail += direction
    return ''.join(''.join(row) for row in fence)

def rail_fence_decrypt(ciphertext, rails):
    """Reconstruct the zigzag pattern and recover plaintext."""
    n = len(ciphertext)
    pattern = []
    rail, direction = 0, 1
    for _ in range(n):
        pattern.append(rail)
        if rail == 0:           direction = 1
        elif rail == rails - 1: direction = -1
        rail += direction
    counts = [pattern.count(r) for r in range(rails)]
    fence, idx = [], 0
    for count in counts:
        fence.append(list(ciphertext[idx:idx+count]))
        idx += count
    rail_idx = [0] * rails
    plaintext = ''
    for r in pattern:
        plaintext += fence[r][rail_idx[r]]
        rail_idx[r] += 1
    return plaintext

def show_fence(plaintext, rails):
    """Display the zigzag fence visually."""
    fence = [[' '] * len(plaintext) for _ in range(rails)]
    rail, direction = 0, 1
    for i, ch in enumerate(plaintext):
        fence[rail][i] = ch
        if rail == 0:           direction = 1
        elif rail == rails - 1: direction = -1
        rail += direction
    print("Rail Fence Visualization:")
    for row in fence:
        print(''.join(row))

# ==================== DEMO ====================
print("=" * 50)
print("   RAIL FENCE CIPHER")
print("=" * 50)
plaintext = "WEAREDISCOVEREDRUNATONCE"
rails = 3

print(f"Plaintext : {plaintext}")
print(f"Rails     : {rails}\n")
show_fence(plaintext, rails)

encrypted = rail_fence_encrypt(plaintext, rails)
decrypted = rail_fence_decrypt(encrypted, rails)

print(f"\nEncrypted : {encrypted}")
print(f"Decrypted : {decrypted}")

---
## Prac 10 — Row (Columnar) Transposition Cipher (Transposition)
Text is filled into a grid **row by row**, then columns are read in the alphabetical order of the keyword letters.

In [ ]:
# ============================================================
# PRAC 10: Row Transposition Cipher (Columnar Transposition)
# ============================================================

def row_transpose_encrypt(plaintext, key):
    """
    Write plaintext in rows of len(key) columns.
    Read columns in the order defined by the sorted key.
    """
    key = key.upper()
    n_cols = len(key)
    padded = plaintext.upper().replace(' ', '')
    while len(padded) % n_cols != 0:
        padded += 'X'
    n_rows = len(padded) // n_cols
    grid = [list(padded[i*n_cols:(i+1)*n_cols]) for i in range(n_rows)]
    order = sorted(range(n_cols), key=lambda x: key[x])
    ciphertext = ''
    for col in order:
        for row in grid:
            ciphertext += row[col]
    return ciphertext

def row_transpose_decrypt(ciphertext, key):
    """Reverse the columnar transposition."""
    key = key.upper()
    n_cols = len(key)
    n_rows = len(ciphertext) // n_cols
    order = sorted(range(n_cols), key=lambda x: key[x])
    cols, idx = {}, 0
    for col in order:
        cols[col] = list(ciphertext[idx:idx+n_rows])
        idx += n_rows
    plaintext = ''
    for r in range(n_rows):
        for c in range(n_cols):
            plaintext += cols[c][r]
    return plaintext

def print_grid(text, key):
    n_cols = len(key)
    padded = text.upper().replace(' ', '')
    while len(padded) % n_cols != 0:
        padded += 'X'
    n_rows = len(padded) // n_cols
    order = sorted(range(n_cols), key=lambda x: key.upper()[x])
    key_order = [order.index(i)+1 for i in range(n_cols)]
    print("Grid:")
    print(' '.join(f"{k}({o})" for k, o in zip(key.upper(), key_order)))
    print('-' * (n_cols * 5))
    for i in range(n_rows):
        print(' '.join(f"  {padded[i*n_cols+j]} " for j in range(n_cols)))

# ==================== DEMO ====================
print("=" * 50)
print("   ROW TRANSPOSITION CIPHER")
print("=" * 50)
plaintext = "WEAREDISCOVEREDFLEEATONCE"
key = "ZEBRAS"

print(f"Plaintext : {plaintext}")
print(f"Key       : {key}\n")
print_grid(plaintext, key)

encrypted = row_transpose_encrypt(plaintext, key)
decrypted = row_transpose_decrypt(encrypted, key)

print(f"\nEncrypted : {encrypted}")
print(f"Decrypted : {decrypted}")

---
## Prac 11 — Product Cipher (Transposition × Transposition)
Combines **two transposition ciphers** in sequence for stronger encryption:
1. **Encrypt:** Rail Fence → Row Transposition
2. **Decrypt:** Row Transposition⁻¹ → Rail Fence⁻¹

In [ ]:
# ============================================================
# PRAC 11: Product Cipher (Rail Fence + Row Transposition)
# ============================================================

# -------- Rail Fence --------
def rf_encrypt(plaintext, rails):
    fence = [[] for _ in range(rails)]
    rail, direction = 0, 1
    for ch in plaintext:
        fence[rail].append(ch)
        if rail == 0:           direction = 1
        elif rail == rails - 1: direction = -1
        rail += direction
    return ''.join(''.join(row) for row in fence)

def rf_decrypt(ciphertext, rails):
    n = len(ciphertext)
    pattern, rail, direction = [], 0, 1
    for _ in range(n):
        pattern.append(rail)
        if rail == 0:           direction = 1
        elif rail == rails - 1: direction = -1
        rail += direction
    counts = [pattern.count(r) for r in range(rails)]
    fence, idx = [], 0
    for count in counts:
        fence.append(list(ciphertext[idx:idx+count]))
        idx += count
    rail_idx = [0] * rails
    plaintext = ''
    for r in pattern:
        plaintext += fence[r][rail_idx[r]]
        rail_idx[r] += 1
    return plaintext

# -------- Row Transposition --------
def rt_encrypt(plaintext, key):
    key = key.upper()
    n_cols = len(key)
    padded = plaintext.upper().replace(' ', '')
    while len(padded) % n_cols != 0:
        padded += 'X'
    n_rows = len(padded) // n_cols
    grid = [list(padded[i*n_cols:(i+1)*n_cols]) for i in range(n_rows)]
    order = sorted(range(n_cols), key=lambda x: key[x])
    return ''.join(grid[r][col] for col in order for r in range(n_rows))

def rt_decrypt(ciphertext, key):
    key = key.upper()
    n_cols = len(key)
    n_rows = len(ciphertext) // n_cols
    order = sorted(range(n_cols), key=lambda x: key[x])
    cols, idx = {}, 0
    for col in order:
        cols[col] = list(ciphertext[idx:idx+n_rows])
        idx += n_rows
    return ''.join(cols[c][r] for r in range(n_rows) for c in range(n_cols))

# -------- Product Cipher --------
def product_encrypt(plaintext, rails, key):
    stage1 = rf_encrypt(plaintext.upper().replace(' ', ''), rails)
    stage2 = rt_encrypt(stage1, key)
    return stage1, stage2

def product_decrypt(ciphertext, rails, key):
    stage1 = rt_decrypt(ciphertext, key)
    stage1_stripped = stage1.rstrip('X')
    stage2 = rf_decrypt(stage1_stripped, rails)
    return stage1_stripped, stage2

# ==================== DEMO ====================
print("=" * 55)
print("   PRODUCT CIPHER  (Rail Fence + Row Transposition)")
print("=" * 55)

plaintext = "WEAREDISCOVEREDFLEEATONCE"
rails = 3
key   = "HACK"

print(f"Plaintext        : {plaintext}")
print(f"Rails            : {rails}")
print(f"Transposition Key: {key}")

print("\n--- ENCRYPTION ---")
after_rail, final_cipher = product_encrypt(plaintext, rails, key)
print(f"After Rail Fence : {after_rail}")
print(f"Final Ciphertext : {final_cipher}")

print("\n--- DECRYPTION ---")
after_row, recovered = product_decrypt(final_cipher, rails, key)
print(f"After Row Transp : {after_row}")
print(f"Recovered Text   : {recovered}")

print("\n--- VERIFICATION ---")
original = plaintext.upper().replace(' ', '')
match = original == recovered[:len(original)]
print(f"Original  : {original}")
print(f"Recovered : {recovered}")
print(f"Match     : {'YES' if match else 'NO'}")